In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
import hvplot.pandas
from rembox_integration_tools import REMboxDataQuery
from rembox_integration_tools.rembox_analysis import StudyColumn, SeriesColumn

hvplot.extension("bokeh")

CLIENT_ID_ENV_VAR = "REMBOX_INT_CLIENT_ID"
CLIENT_PWD_ENV_VAR = "REMBOX_INT_CLIENT_PWD"
TOKEN_URI = "https://autoqa.vll.se/dpqaauth/connect/token"
API_URI = "https://rembox.vll.se/api"
ORIGIN_URI = "https://rembox.vll.se"

rembox = REMboxDataQuery(
    client_id_environment_variable=CLIENT_ID_ENV_VAR,
    client_secret_environment_variable=CLIENT_PWD_ENV_VAR,
    token_uri=TOKEN_URI,
    api_uri=API_URI,
    origin_uri=ORIGIN_URI,
    verify_ssl_cert=False
)

valid_study_columns = StudyColumn()
valid_series_columns = SeriesColumn()

In [ ]:
def get_data_from_REMbox(rembox: REMboxDataQuery) -> tuple[pd.DataFrame, pd.DataFrame]:
    valid_study_columns = StudyColumn()
    valid_series_columns = SeriesColumn()

    
    rembox.filter_options.set_inclusive_tags(
        machine_types=['DX'],
        hospitals=['NUS']
    )


   # exclude Acquisition protocols - positionsgenmomlysning
    rembox.filter_options.set_exclusive_tags(
        acquisition_protocols=['CP_Positioning',
                               'CP_Barn buk',
                               'CP_antiiso',
                               'Småskelett']
    )

    # about four month data
    rembox.filter_options.study_time_interval_start_date = (dt.datetime.now() - dt.timedelta(days=7)).strftime("%Y-%m-%dT%H:%M:%SZ")
    rembox.filter_options.study_time_interval_end_date = dt.datetime.now().strftime("%Y-%m-%dT%H:%M:%SZ")


    rembox.add_columns(
        columns=[
            valid_study_columns.StudyDateTime,
            valid_study_columns.StudyInstanceUID,
            valid_study_columns.StudyId,
            valid_study_columns.Machine,
            valid_study_columns.AccessionNumber,
            valid_study_columns.StudyDescription,
            valid_study_columns.PatientAge,
            valid_study_columns.DoseAreaProductTotal,
            valid_study_columns.PatientDbId,
            valid_study_columns.TotalNumberOfRadiographicFrames,
            valid_series_columns.AcquisitionProtocol,
            valid_series_columns.DateTimeStarted,
            valid_series_columns.DoseAreaProduct,
            valid_series_columns.ExposureIndex
        ]
    )

    return rembox.run_query()

In [ ]:
study_data, series_data = get_data_from_REMbox(rembox=rembox)
data = series_data.merge(study_data, on=['studyInstanceUID'], how="left")

In [ ]:
high_exposure_index = data[data.exposureIndex>750]
low_exposure_index = data[data.exposureIndex<80]

In [ ]:
low_exposure_index_summery = low_exposure_index[['acquisitionProtocol', 'exposureIndex', 'accessionNumber_y', 'dateTimeStarted', 'machine', 'totalNumberOfRadiographicFrames']]

In [ ]:
high_exposure_index_summery = high_exposure_index[['acquisitionProtocol', 'exposureIndex', 'accessionNumber_x', 'dateTimeStarted', 'machine']]